# KV Cache from Scratch

**Companion notebook for: KV Cache: Intuition, Implementation, Production**

Three sections matching the article:
1. **The Problem**: standard attention recomputes everything every step
2. **The Fix**: three lines of code, measured speedup
3. **Production tricks**: INT8 quantization and Grouped Query Attention

No GPU needed. The whole notebook runs on CPU in under 10 seconds.

> The `Head` class here is the same single-head causal self-attention pattern used in GPT-2. Real transformer code, just one head instead of many.

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

SEED = 2026
torch.manual_seed(SEED)
print(f"PyTorch: {torch.__version__}")

PyTorch: 2.11.0


## Section 1: The Problem

Every time you generate a new token, a transformer runs attention over the **full sequence so far**.

For a sequence of length T, that means:
- T key vectors
- T value vectors
- a T×T attention matrix

Generate 200 tokens and you have run attention 200 times. Each time over a longer sequence than before. The work compounds.

Here is what standard single-head attention looks like. The thing to watch is where K and V get computed.

In [22]:
class Head(nn.Module):
    """Single-head causal self-attention. Standard. No caching.
    more details here: 
    https://github.com/bhuvanchennoju/GPT-from-scratch/blob/master/src/multiheaded_attention_bigram/model.py
    """

    def __init__(self, n_embed, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key   = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        # These three lines are the problem.
        # K, Q, V get computed for ALL T tokens, every single time forward() is called.
        k = self.key(x) # (B, T,head_size)
        q = self.query(x)# (B, T,head_size)
        v = self.value(x)# (B, T,head_size)

        wei = q @ k.transpose(-2, -1) * C**(-0.5) # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v # (B, T, head_size)

### Watch generation get slower

Let me simulate what actually happens during decoding. Start with a short prompt, generate one token at a time, feed the entire growing sequence back through attention every step.

In [23]:
@torch.no_grad()
def generate_no_cache(head, n_embed, prompt_len=10, n_new_tokens=10):
    """Simulate token generation with no cache.
    Each step feeds the full growing sequence through attention."""
    lm_head = nn.Linear(n_embed, n_embed)
    x = torch.randn(1, prompt_len, n_embed)

    for _ in range(n_new_tokens):
        recom = head(x) # recomputes K,V for ALL past tokens
        next_tok = lm_head(recom[:,-1:,:]) # predict from the last position
        x = torch.cat([x, next_tok], dim=1)   # sequence grows by 1
    return x.shape[1]

In [24]:
n_embed,head_size,block_size = 64,64,512 #embeding size,attention head size,context window size
head = Head(n_embed, head_size, block_size)

final_len = generate_no_cache(head,n_embed,prompt_len=10,n_new_tokens=10)
print(f"Final sequence length:{final_len}")

Final sequence length:20


In [25]:
print("Time per attention call as the sequence grows (no cache):")
print(f" {'seq_len':>8}  {'avg ms':>10}")
print("  " + "-" * 50)

for seq_len in [10, 50, 100, 200, 400]:
    times = []
    for _ in range(30):
        x = torch.randn(1, seq_len, n_embed)
        t0 = time.perf_counter()
        with torch.no_grad():
            head(x)
        times.append(time.perf_counter() - t0)
    avg_ms = sum(times) / len(times) * 1000
    print(f"  {seq_len:8d}  {avg_ms:8.3f}ms")

Time per attention call as the sequence grows (no cache):
  seq_len      avg ms
  --------------------------------------------------
        10     0.081ms
        50     0.065ms
       100     0.074ms
       200     0.166ms
       400     0.312ms


Each call is slower than the last. That is O(n²) attention scaling with the sequence length. Multiply that by every step of generation and the cost compounds fast.

## Section 2: The Fix

The problem is that K and V get recomputed for every past token at every step. But those past K and V values do not change. The token already happened. Computing them again is pure waste.

**The fix:** compute K and V once per token, save them, and reuse them on every future step.

Q is never cached. Q only matters for the current token making the prediction. Once that prediction is made, Q for that token is done.

The change to `forward()` is three lines. Everything else stays the same.

In [26]:
class HeadWithCache(nn.Module):
    """Same as Head — with KV cache added in 3 lines."""

    def __init__(self, n_embed, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key   = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None): # accept cached K, V
        B, T, C = x.shape
        k = self.key(x)# only for the NEW tokens in x
        q = self.query(x)
        v = self.value(x)

        if past_kv is not None: #prepend past K, V
            k = torch.cat([past_kv[0], k], dim=1)
            v = torch.cat([past_kv[1], v], dim=1)

        T_full = k.shape[1]
        wei = q @ k.transpose(-2, -1) * C**(-0.5)
        if T > 1:  # prefill: apply causal mask; decode (T=1): new token sees all past
            wei = wei.masked_fill(self.tril[:T, :T_full] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v

        return out, (k, v) #return updated cache

Three changes total:

1. `forward` now accepts `past_kv`, the K and V from all previous steps.
2. If the cache exists, prepend it to the freshly computed K and V before running attention.
3. Return the updated cache so the caller can pass it back next step.

**Prefill phase:** pass the full prompt with `past_kv=None`. Cache is built from scratch.

**Decode phase:** pass one token at a time. Cache grows by one row per step. Past K and V are never recomputed.

Watch the cache grow:

In [27]:
@torch.no_grad()
def generate_with_cache(head, n_embed, prompt_len=10, n_new_tokens=10):
    """Generate with KV cache.
    Prefill: process the full prompt once.
    Decode: feed only the new token each step."""
    lm_head = nn.Linear(n_embed, n_embed)
    prompt = torch.randn(1, prompt_len, n_embed)

    # Prefill — process the whole prompt, build the initial cache
    _, past_kv = head(prompt, past_kv=None)
    print(f"After prefill   cache K shape: {past_kv[0].shape}")

    # Decode — one token at a time, cache grows by one each step
    for step in range(n_new_tokens):
        new_tok = torch.randn(1, 1, n_embed) # only 1 token now
        out, past_kv = head(new_tok, past_kv=past_kv)
        if step < 3:
            print(f"After step {step+1:2d}    cache K shape: {past_kv[0].shape}")
    print(f"Final cache K shape: {past_kv[0].shape}")
    return past_kv[0].shape[1]

head_cache = HeadWithCache(n_embed, head_size, block_size)
generate_with_cache(head_cache, n_embed, prompt_len=10, n_new_tokens=10)

After prefill   cache K shape: torch.Size([1, 10, 64])
After step  1    cache K shape: torch.Size([1, 11, 64])
After step  2    cache K shape: torch.Size([1, 12, 64])
After step  3    cache K shape: torch.Size([1, 13, 64])
Final cache K shape: torch.Size([1, 20, 64])


20

### Benchmark: cache vs no cache

Now the actual comparison. Same model, same sequence lengths. One uses cache, one does not.

In [28]:
print("Benchmark: cached vs no cache:")
print(f"  {'seq_len':>8}  {'no cache':>12}  {'with cache':>12}  {'speedup':>10}")
print("  " + "-" * 50)

head_base  = Head(n_embed, head_size, block_size)
head_cache = HeadWithCache(n_embed, head_size, block_size)

for seq_len in [50,100,200,500]:

    # No cache: full growing sequence at every step
    t0 = time.perf_counter()
    with torch.no_grad():
        for i in range(seq_len):
            x = torch.randn(1, i + 1, n_embed)
            head_base(x)
    no_cache_s = time.perf_counter() - t0

    # With cache: prefill once, then one token per step
    t0 = time.perf_counter()
    with torch.no_grad():
        prompt = torch.randn(1, 1, n_embed)
        _, past_kv = head_cache(prompt)
        for _ in range(seq_len - 1):
            tok = torch.randn(1, 1, n_embed)
            _, past_kv = head_cache(tok, past_kv=past_kv)
    cache_s = time.perf_counter() - t0

    print(f"  {seq_len:8d}  {no_cache_s:10.3f}s  {cache_s:10.3f}s  {no_cache_s/cache_s:8.1f}x")

Benchmark: cached vs no cache:
   seq_len      no cache    with cache     speedup
  --------------------------------------------------
        50       0.005s       0.002s       2.7x
       100       0.009s       0.003s       2.9x
       200       0.028s       0.007s       4.2x
       500       0.164s       0.020s       8.0x


The speedup grows with sequence length.
* No-cache is O(n²), total work scales with the sum of every prior length.
* Cached is O(n), each step is constant work, just an append plus a lookup.

## Section 3: Production tricks

Implementing a KV cache is easy. Running it at scale is where things get interesting.

Here are two techniques production systems layer on top:

| Technique | What it does | Saving |
|---|---|---|
| INT8 quantization | Store K and V in 8-bit instead of 32-bit | 4x memory |
| Grouped Query Attention | Multiple Q heads share one K and V head | 4-8x KV cache size |

These are additive. A real production deployment uses both.

### 1. Quantization

A 70B model serving 1000 tokens generates around 2.5 GB of KV cache in FP32. INT8 cuts that to about 625 MB. Same data, smaller numbers.

How it works in two functions: scale the tensor so its biggest value fits in INT8 range, round, store. Reverse on read.

In [29]:
def quantize_cache(tensor: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Quantize a float tensor to INT8 with a single scale factor."""
    scale = tensor.abs().max() / 127
    quantized = (tensor / scale).round().clamp(-128, 127).to(torch.int8)
    return quantized, scale


def dequantize_cache(quantized: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    """Recover the float tensor (with some precision loss)."""
    return quantized.float() * scale


# 1000 tokens of d_model=512 cache (scaled-down 70B example)
cache_fp32 = torch.randn(1, 1000, 512)
cache_int8, scale = quantize_cache(cache_fp32)
cache_back = dequantize_cache(cache_int8, scale)

fp32_mb = cache_fp32.nelement() * cache_fp32.element_size() / 1e6
int8_mb = cache_int8.nelement() * cache_int8.element_size() / 1e6
max_err = (cache_fp32 - cache_back).abs().max().item()

print(f"FP32 cache size:{fp32_mb:.2f} MB")
print(f"INT8 cache size:{int8_mb:.2f} MB")
print(f"Compression:{fp32_mb / int8_mb:.0f}x")
print(f"Max absolute error: {max_err:.5f}")

FP32 cache size:2.05 MB
INT8 cache size:0.51 MB
Compression:4x
Max absolute error: 0.01935


### 2. Grouped Query Attention (GQA)

Standard multi-head attention gives each query head its own K and V head. If you have 64 Q heads, you have 64 K heads and 64 V heads.

GQA shares K and V across groups. Llama 2 70B has 64 Q heads and 8 KV heads. That is an 8x smaller KV cache at the architecture level. No quality loss for most tasks.

**One thing that trips people up:** Q, K, V are packed as `(batch * n_heads, seq_len, d_head)`. The heads live in the **batch** dimension, not the sequence dimension. So expanding K and V to match Q means repeating along `dim=0` (heads), not `dim=1` (sequence positions). Get this wrong and you silently corrupt attention.

In [30]:
def multiheadattention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Standard scaled dot-product attention."""
    scale = math.sqrt(q.shape[-1]) #sqrt of 64--> 8
    scores = torch.bmm(q, k.transpose(1, 2)) / scale # q: (B, T_q, D), k: (B, T_kv, D) --> scores: (B, T_q, T_kv);
                                                    #(8,20,64) @ (8,64,20) = (8, 20, 20)--> 20 x 20 attention scores per head
    return torch.bmm(F.softmax(scores, dim=-1), v) # (B, T_q, T_kv) @ (B, T_kv, D) --> (B, T_q, D); 
                                                   #(8,20,20) @ (8,20,64) = (8, 20, 64) --> output per head 


def grouped_query_attention(q: torch.Tensor,
                            k: torch.Tensor,
                            v: torch.Tensor,
                            n_q_heads: int,
                            n_kv_heads: int) -> torch.Tensor:
    """GQA: expand K and V along the heads dimension to match Q heads."""
    group_size = n_q_heads // n_kv_heads
    k = k.repeat_interleave(group_size,dim=0)# (2,20,64)--> (8,20,64) # heads, not sequence
    v = v.repeat_interleave(group_size,dim=0)# (2,20,64)--> (8,20,64)
    return multiheadattention(q,k,v)


# 4:1 ratio, like Llama 2 70B at smaller scale
batch, seq_len, d_head = 1, 20, 64
n_q_heads, n_kv_heads = 8, 2

q = torch.randn(batch * n_q_heads,seq_len,d_head)# (8, 20, 64)
k = torch.randn(batch * n_kv_heads,seq_len,d_head)# (2, 20, 64) — only 2 KV heads
v = torch.randn(batch * n_kv_heads,seq_len,d_head)# (2, 20, 64)

gqa_ = grouped_query_attention(q, k, v, n_q_heads, n_kv_heads)
print(f"GQA output shape: {gqa_.shape}") #(8, 20, 64)

mha_kv = 2 * n_q_heads  * seq_len * d_head
gqa_kv = 2 * n_kv_heads * seq_len * d_head
print(f"\nMHA KV cache elements: {mha_kv:,}")
print(f"GQA KV cache elements: {gqa_kv:,}")
print(f"Reduction: {mha_kv // gqa_kv}x")

GQA output shape: torch.Size([8, 20, 64])

MHA KV cache elements: 20,480
GQA KV cache elements: 5,120
Reduction: 4x


## Summary

| Technique | What it changes | Typical gain |
|---|---|---|
| KV cache | Skip K and V recompute for past tokens | 5-15x latency |
| INT8 quantization | Store K and V in 8-bit | 4x memory |
| Grouped Query Attention | Share K and V heads across query groups | 4-8x KV cache size |


These are additive.

**Coming next in this series:**
- Autoregressive decoding
- Attention mechanism
- Paged attention and vLLM

Full article: [bhuvanchennoju.com/posts/kv-cache](https://bhuvanchennoju.com)